# FinalBaselines + AttentionBaseline altyapısı

In [ ]:
import platform, subprocess
print("Platform:", platform.platform())
# CPU modeli
try:
    print(subprocess.check_output("lscpu | grep -E 'Model name|Socket|Core|Thread|MHz'", shell=True).decode())
except: pass
# bellek
try:
    print(subprocess.check_output("free -h | head -2", shell=True).decode())
except: pass
import torch
print("torch:", torch.__version__, "| threads:", torch.get_num_threads())

Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Thread(s) per core:                      2
Core(s) per socket:                      1
Socket(s):                               1

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.2Gi       8.0Gi       2.0Mi       3.5Gi        11Gi

torch: 2.11.0+cpu | threads: 1


In [ ]:
# EDGE-REALISTIC (single-core CPU)

import os
os.environ["OMP_NUM_THREADS"]      = "1"   # numpy/BLAS tek çekirdek
os.environ["MKL_NUM_THREADS"]      = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"]  = "1"

import numpy as np, pandas as pd, time, torch
torch.set_num_threads(1)                    # torch tek çekirdek
#DEV = torch.device("cpu")                   # attention'ı CPU'ya zorla (edge)
RATE = ["s_fanout_rate","s_flows_per_dst","s_dst_entropy_norm"]
#print("Edge-realistic setting: single CPU core, no GPU. DEV =", DEV)

torch.manual_seed(42); np.random.seed(42)
DEV="cuda" if torch.cuda.is_available() else "cpu"
print("torch",torch.__version__,"| device",DEV)

torch 2.11.0+cpu | device cpu


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import time, tracemalloc
from numpy.linalg import pinv
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

np.random.seed(42)
DATA="/data/"
SEEDS=(42,1,2,3,4); RS=42; FPR_GRID=(0.05,0.10,0.20)
UAV_CFG={"path":DATA+"UAVIDS-2025.csv","label_col":"label","normal":"Normal Traffic",
  "attacks":["Blackhole Attack","Flooding Attack","Sybil Attack","Wormhole Attack"],
  "leak_clean":["FlowID","SrcAddr","DstAddr","Protocol"]}
ID_CFG={"src":"SrcAddr","dst":"DstAddr"}; TARGET="Sybil Attack"
RATE=["s_fanout_rate","s_flows_per_dst","s_dst_entropy_norm"]
print("config ok")

config ok


## FinalBaselines altyapısı

In [ ]:
def load(cfg): return pd.read_csv(cfg["path"],low_memory=False).reset_index(drop=True)
def source_rate_features(df,idcfg):
    src=idcfg["src"]; dst=idcfg["dst"]
    s=df[src].astype(str).fillna("NA").values; dd=df[dst].astype(str).fillna("NA").values
    tmp=pd.DataFrame({"s":s,"d":dd}); g=tmp.groupby("s")
    flowcount=g["s"].transform("size").astype(float).values; fanout=g["d"].transform("nunique").astype(float).values
    ent_map={}
    for k,gg in tmp.groupby("s"):
        vc=gg["d"].value_counts().values.astype(float); p=vc/vc.sum(); ent_map[k]=float(-(p*np.log(p+1e-12)).sum())
    ent=np.array([ent_map[x] for x in s]); fc=np.clip(flowcount,1,None); fo=np.clip(fanout,1,None)
    return pd.DataFrame({"s_fanout_rate":fanout/fc,"s_flows_per_dst":flowcount/fo,
                         "s_dst_entropy_norm":ent/np.log(np.clip(fanout,2,None))}).fillna(0.0).reset_index(drop=True)
def feats_perflow(df,cfg):
    # CLEAN per-flow: drop leak_clean, label, AND any source-rate columns
    drop=set(cfg["leak_clean"])|{cfg["label_col"]}|set(RATE)
    X=df.drop(columns=[c for c in df.columns if c in drop],errors="ignore").copy()
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]): X[c]=LabelEncoder().fit_transform(X[c].astype(str))
    return X.astype(float).reset_index(drop=True)
def loaco_split(df,label_col,target,seed,val=0.3,test=0.3):
    rng=np.random.default_rng(seed)
    dft=df[df[label_col].astype(str)==target]; dfk=df[df[label_col].astype(str)!=target]
    idx=rng.permutation(len(dfk)); nv=int(len(idx)*val)
    valk=dfk.iloc[idx[:nv]]; tr=dfk.iloc[idx[nv:]]; nt=int(len(tr)*test)
    return tr.iloc[nt:], valk, pd.concat([tr.iloc[:nt],dft])
def ae_(n,seed):
    b=max(2,n//4)
    return MLPRegressor(hidden_layer_sizes=(max(8,n//2),b,max(8,n//2)),max_iter=150,early_stopping=True,n_iter_no_change=8,random_state=seed)
def ae_err(ae,X):
    r=ae.predict(X); r=r.reshape(-1,1) if r.ndim==1 else r; return np.mean((X-r)**2,1)
def fit_mahal(Xn):
    mu=Xn.mean(0); cov=np.cov(Xn.T)+1e-6*np.eye(Xn.shape[1]); return mu,pinv(cov)
def mahal(X,mu,P):
    d=X-mu; return np.einsum('ij,jk,ik->i',d,P,d)
def Z(A,Bv,Be):
    imp=SimpleImputer(strategy="mean").fit(A); sca=StandardScaler().fit(imp.transform(A))
    return sca.transform(imp.transform(A)),sca.transform(imp.transform(Bv)),sca.transform(imp.transform(Be))
print("helpers ok")

helpers ok


In [ ]:
def _Z(A, Be):               # impute+scale on A (train), apply to Be (test)
    imp=SimpleImputer(strategy="mean").fit(A); sc=StandardScaler().fit(imp.transform(A))
    return sc.transform(imp.transform(A)), sc.transform(imp.transform(Be))

## AttentionBaseline Altyapısı

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import torch, torch.nn as nn
from numpy.linalg import pinv
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
torch.manual_seed(42); np.random.seed(42)
DEV="cuda" if torch.cuda.is_available() else "cpu"
print("torch",torch.__version__,"| device",DEV)

torch 2.11.0+cpu | device cpu


In [ ]:
EPOCHS=40        #@param
SUBSAMPLE=0      #@param
SUBSAMPLE=None if SUBSAMPLE in (0,None) else int(SUBSAMPLE)
UAV_CFG={"path":DATA+"UAVIDS-2025.csv","label_col":"label","normal":"Normal Traffic",
  "attacks":["Blackhole Attack","Flooding Attack","Sybil Attack","Wormhole Attack"],
  "leak_clean":["FlowID","SrcAddr","DstAddr","Protocol"]}
ID_CFG={"src":"SrcAddr","dst":"DstAddr"}; TARGET="Sybil Attack"
RATE=["s_fanout_rate","s_flows_per_dst","s_dst_entropy_norm"]

def load(cfg):
    df=pd.read_csv(cfg["path"],low_memory=False).reset_index(drop=True)
    if SUBSAMPLE:
        df=pd.concat([g.sample(min(len(g),SUBSAMPLE),random_state=RS) for _,g in df.groupby(cfg["label_col"])]).reset_index(drop=True)
    return df
def source_rate_features(df,idcfg):
    src=idcfg["src"]; dst=idcfg["dst"]
    s=df[src].astype(str).fillna("NA").values; dd=df[dst].astype(str).fillna("NA").values
    tmp=pd.DataFrame({"s":s,"d":dd}); g=tmp.groupby("s")
    flowcount=g["s"].transform("size").astype(float).values; fanout=g["d"].transform("nunique").astype(float).values
    ent_map={}
    for k,gg in tmp.groupby("s"):
        vc=gg["d"].value_counts().values.astype(float); p=vc/vc.sum(); ent_map[k]=float(-(p*np.log(p+1e-12)).sum())
    ent=np.array([ent_map[x] for x in s]); fc=np.clip(flowcount,1,None); fo=np.clip(fanout,1,None)
    return pd.DataFrame({"s_fanout_rate":fanout/fc,"s_flows_per_dst":flowcount/fo,
                         "s_dst_entropy_norm":ent/np.log(np.clip(fanout,2,None))}).fillna(0.0).reset_index(drop=True)
def feats_perflow(df,cfg):
    drop=set(cfg["leak_clean"])|{cfg["label_col"]}|set(RATE)
    X=df.drop(columns=[c for c in df.columns if c in drop],errors="ignore").copy()
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]): X[c]=LabelEncoder().fit_transform(X[c].astype(str))
    return X.astype(float).reset_index(drop=True)
def loaco_split(df,label_col,target,seed,val=0.3,test=0.3):
    rng=np.random.default_rng(seed)
    dft=df[df[label_col].astype(str)==target]; dfk=df[df[label_col].astype(str)!=target]
    idx=rng.permutation(len(dfk)); nv=int(len(idx)*val)
    valk=dfk.iloc[idx[:nv]]; tr=dfk.iloc[idx[nv:]]; nt=int(len(tr)*test)
    return tr.iloc[nt:], valk, pd.concat([tr.iloc[:nt],dft])
def fit_mahal(Xn):
    mu=Xn.mean(0); cov=np.cov(Xn.T)+1e-6*np.eye(Xn.shape[1]); return mu,pinv(cov)
def mahal(X,mu,P):
    d=X-mu; return np.einsum('ij,jk,ik->i',d,P,d)
def scale(A,Bv,Be):
    imp=SimpleImputer(strategy="mean").fit(A); sc=StandardScaler().fit(imp.transform(A))
    return sc.transform(imp.transform(A)),sc.transform(imp.transform(Bv)),sc.transform(imp.transform(Be))
print("helpers ok")

helpers ok


In [ ]:
class TabAttnAE(nn.Module):
    def __init__(self,n_features,d=32,nhead=4,nlayers=2,latent=8):
        super().__init__()
        self.n=n_features; self.d=d
        self.val_proj=nn.Linear(1,d)
        self.feat_emb=nn.Parameter(torch.randn(n_features,d)*0.02)
        enc=nn.TransformerEncoderLayer(d_model=d,nhead=nhead,dim_feedforward=2*d,batch_first=True,dropout=0.0)
        self.encoder=nn.TransformerEncoder(enc,num_layers=nlayers)
        self.to_latent=nn.Linear(n_features*d,latent)
        self.from_latent=nn.Linear(latent,n_features*d)
        self.out=nn.Linear(d,1)
    def forward(self,x):
        B=x.size(0)
        tok=self.val_proj(x.unsqueeze(-1))+self.feat_emb.unsqueeze(0)   # [B,n,d]
        z=self.encoder(tok)                                            # [B,n,d]
        lat=self.to_latent(z.reshape(B,-1))
        h=self.from_latent(lat).reshape(B,self.n,self.d)
        return self.out(h).squeeze(-1)                                 # [B,n]

def train_attn_ae(Xtr,seed,epochs=EPOCHS,bs=512,lr=1e-3):
    torch.manual_seed(seed)
    n=Xtr.shape[1]; d=32; nhead=4 if n>=4 else 1
    m=TabAttnAE(n,d=d,nhead=nhead).to(DEV)
    opt=torch.optim.Adam(m.parameters(),lr=lr)
    X=torch.tensor(Xtr,dtype=torch.float32)
    ds=torch.utils.data.TensorDataset(X)
    dl=torch.utils.data.DataLoader(ds,batch_size=bs,shuffle=True)
    lossf=nn.MSELoss()
    m.train()
    for ep in range(epochs):
        for (xb,) in dl:
            xb=xb.to(DEV); opt.zero_grad()
            rec=m(xb); loss=lossf(rec,xb); loss.backward(); opt.step()
    return m
@torch.no_grad()
def attn_err(m,X):
    m.eval(); xb=torch.tensor(X,dtype=torch.float32).to(DEV)
    rec=m(xb); return ((xb-rec)**2).mean(1).cpu().numpy()
print("model ok")

model ok


In [ ]:
def train_attn_ae_bottleneck(Xtr, seed, epochs=20, bs=512, lr=1e-3, latent=2):
    torch.manual_seed(seed)
    n=Xtr.shape[1]; d=32; nhead=4 if n>=4 else 1
    lat=min(latent, max(1, n-1))                 # latent GİRDİDEN KÜÇÜK (gerçek sıkıştırma)
    m=TabAttnAE(n, d=d, nhead=nhead, latent=lat).to(DEV)
    opt=torch.optim.Adam(m.parameters(), lr=lr)
    X=torch.tensor(Xtr, dtype=torch.float32)
    dl=torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X), batch_size=bs, shuffle=True)
    lossf=nn.MSELoss(); m.train()
    for _ in range(epochs):
        for (xb,) in dl:
            xb=xb.to(DEV); opt.zero_grad()
            loss=lossf(m(xb), xb); loss.backward(); opt.step()
    return m, lat


def run_attn_rate_fair(seeds=SEEDS, grid=FPR_GRID, epochs=20, latent=2):
    cfg=UAV_CFG; idcfg=ID_CFG; lab=cfg["label_col"]; normal=cfg["normal"]; tgt=TARGET; src=idcfg["src"]
    df=load(cfg); df=pd.concat([df, source_rate_features(df, idcfg)], axis=1)
    roc=[]; op=[]; null_auc=[]; trec=[]; used_lat=None
    for seed in seeds:
        tr,valk,test_=loaco_split(df,lab,tgt,seed); test_=test_.reset_index(drop=True)
        ytr=tr[lab].astype(str).values; yv=valk[lab].astype(str).values; ye=test_[lab].astype(str).values
        is_t=(ye==tgt); is_n=(ye==normal); ka=(~is_t)&(~is_n)
        seen=is_t & test_[src].astype(str).isin(set(tr[src].astype(str))).values; sdj=~seen
        Ar=tr[RATE].reset_index(drop=True); Bvr=valk[RATE].reset_index(drop=True); Ber=test_[RATE].reset_index(drop=True)
        Za,Zv,Ze=scale(Ar,Bvr,Ber)
        # gercek bottleneck modeli
        m,lat=train_attn_ae_bottleneck(Za,seed,epochs=epochs,latent=latent); used_lat=lat
        e=attn_err(m,Ze); base=attn_err(m,Zv)[yv==normal]; trec.append(attn_err(m,Za).mean())
        roc.append(roc_auc_score(is_t[sdj].astype(int), e[sdj]))
        opf={}
        for f in grid:
            thr=np.percentile(base,100*(1-f))
            opf[f]=dict(det=float(np.mean((e[sdj]>thr)[is_t[sdj]])),
                        fpr_normal=float(np.mean((e>thr)[is_n])),
                        fpr_known=float(np.mean((e>thr)[ka])))
        op.append(opf)
        # null kontrolu (ayni bottleneck ile)
        rng=np.random.default_rng(seed); Zp=Za.copy()
        for j in range(Zp.shape[1]): Zp[:,j]=Zp[rng.permutation(Zp.shape[0]),j]
        mp,_=train_attn_ae_bottleneck(Zp,seed,epochs=epochs,latent=latent)
        null_auc.append(roc_auc_score(is_t[sdj].astype(int), attn_err(mp,Ze)[sdj]))
    r=np.array(roc); na=np.array(null_auc)
    print(f"=== attn_ae_rate (FAIR bottleneck: latent={used_lat}<3, epochs={epochs}) ===")
    print(f"  source-disjoint ROC = {r.mean():.3f}\u00b1{r.std():.3f}")
    print(f"  mean train recon error = {np.mean(trec):.4f}   (0.0000 ise hala sikismiyor demektir)")
    print(f"  null/permuted-train ROC = {na.mean():.3f}\u00b1{na.std():.3f}")
    print("  operating points (source-disjoint, mean):")
    for f in grid:
        D=pd.DataFrame([o[f] for o in op]).mean()
        print(f"    fpr={f:.2f} -> det={D['det']:.3f}  fpr_normal={D['fpr_normal']:.3f}  fpr_known={D['fpr_known']:.3f}")
    print("\n  Karsilastirma: mahal_rate_sdj (bizim) = 0.907 | onceki latent=8 attn = 0.996")
    print("  Okuma: ROC korunursa -> saglam co-method; belirgin duserse -> onceki 0.996 bottleneck-yoklugu etkisiydi")
    return roc, op, null_auc

## complexity/training time/inference latency/memory

In [ ]:
# ---------- R1.1: complexity (edge-realistic, single-core CPU) ----------
def complexity(seeds=SEEDS):
    cfg=UAV_CFG; lab=cfg["label_col"]; normal=cfg["normal"]; tgt=TARGET
    df=load(cfg); df=pd.concat([df,source_rate_features(df,ID_CFG)],axis=1)
    rows=[]; nparam=None
    for seed in seeds:
        tr,valk,test_=loaco_split(df,lab,tgt,seed); test_=test_.reset_index(drop=True)
        ytr=tr[lab].astype(str).values; bmask=(ytr==normal)
        A=tr[RATE].reset_index(drop=True); B=test_[RATE].reset_index(drop=True)
        Za,Ze=_Z(A,B); Xb=Za[bmask]; n_test=len(Ze)

        # Mahalanobis: fit (closed-form) + inference
        t0=time.perf_counter(); mu,P=fit_mahal(Xb); t_fit_m=time.perf_counter()-t0
        t0=time.perf_counter(); _=mahal(Ze,mu,P); t_inf_m=time.perf_counter()-t0

        # Attention: fit + inference (CPU)
        """Za2,_,Ze2=scale(A,A,B)
        t0=time.perf_counter(); m,_=train_attn_ae_bottleneck(Za2[bmask],seed,epochs=20,latent=2)
        t_fit_a=time.perf_counter()-t0
        t0=time.perf_counter();"""

        Za2,_,Ze2 = scale(A,A,B)
        t0=time.perf_counter()
        m,_ = train_attn_ae_bottleneck(Za2, seed, epochs=20, latent=2)   # Za2 (tüm known), bmask YOK
        t_fit_a=time.perf_counter()-t0

        _=attn_err(m,Ze2); t_inf_a=time.perf_counter()-t0
        if nparam is None:
            nparam=sum(p.numel() for p in m.parameters())

        rows.append({"n_test":n_test,
                     "maha_fit_ms":t_fit_m*1e3, "maha_inf_ms":t_inf_m*1e3,
                     "attn_fit_s":t_fit_a,      "attn_inf_ms":t_inf_a*1e3})
    R=pd.DataFrame(rows); nt=R.n_test.mean()
    print("=== Complexity (edge-realistic, single CPU core, mean over seeds) ===")
    print(f"  Mahalanobis  | train {R.maha_fit_ms.mean():8.2f} ms (closed-form) | "
          f"inference {R.maha_inf_ms.mean():7.2f} ms  ({R.maha_inf_ms.mean()/nt*1e3:6.2f} µs/flow)")
    print(f"  Attention AE | train {R.attn_fit_s.mean():8.2f} s  (20 epochs)    | "
          f"inference {R.attn_inf_ms.mean():7.2f} ms  ({R.attn_inf_ms.mean()/nt*1e3:6.2f} µs/flow)")
    print(f"  Attention parameters: {nparam:,}")
    print(f"  Test set size: {int(nt):,} flows | Mahalanobis: no iterative training, O(d^2) per flow.")
    return R
complexity()

=== Complexity (edge-realistic, single CPU core, mean over seeds) ===
  Mahalanobis  | train     9.98 ms (closed-form) | inference    1.87 ms  (  0.04 µs/flow)
  Attention AE | train    41.71 s  (20 epochs)    | inference 42416.27 ms  (949.42 µs/flow)
  Attention parameters: 17,763
  Test set size: 44,676 flows | Mahalanobis: no iterative training, O(d^2) per flow.


,n_test,maha_fit_ms,maha_inf_ms,attn_fit_s,attn_inf_ms
0,44676,44.942485,6.596851,52.822521,53298.397065
1,44676,1.231059,0.710920,39.477223,40652.542300
2,44676,1.212878,0.690636,37.658832,38191.624459
3,44676,1.269868,0.683920,39.710728,40208.296434
4,44676,1.247680,0.691792,38.885221,39730.466016


## robustness: noise, imbalance

In [ ]:
# ---------- R1.2a: noise robustness (standardized-space Gaussian noise) ----------
def robustness_noise(seeds=SEEDS, levels=(0.0,0.05,0.10,0.20,0.30)):
    cfg=UAV_CFG; lab=cfg["label_col"]; normal=cfg["normal"]; tgt=TARGET; src=ID_CFG["src"]
    df=load(cfg); df=pd.concat([df,source_rate_features(df,ID_CFG)],axis=1)
    res={L:{"maha":[],"attn":[]} for L in levels}
    for seed in seeds:
        tr,valk,test_=loaco_split(df,lab,tgt,seed); test_=test_.reset_index(drop=True)
        ytr=tr[lab].astype(str).values; ye=test_[lab].astype(str).values
        is_t=(ye==tgt); bmask=(ytr==normal)
        seen=is_t & test_[src].astype(str).isin(set(tr[src].astype(str))).values; sdj=~seen
        yb=is_t[sdj].astype(int)
        A=tr[RATE].reset_index(drop=True); B=test_[RATE].reset_index(drop=True)
        Za,Ze=_Z(A,B); mu,P=fit_mahal(Za[bmask])
        #Za2,_,Ze2=scale(A,A,B); m,_=train_attn_ae_bottleneck(Za2[bmask],seed,epochs=20,latent=2)

        Za2,_,Ze2 = scale(A,A,B)
        m,_ = train_attn_ae_bottleneck(Za2, seed, epochs=20, latent=2)

        rng=np.random.default_rng(seed)
        for L in levels:
            Zen  = Ze  + rng.normal(0,L,Ze.shape)
            Ze2n = Ze2 + rng.normal(0,L,Ze2.shape)
            res[L]["maha"].append(roc_auc_score(yb, mahal(Zen,mu,P)[sdj]))
            res[L]["attn"].append(roc_auc_score(yb, attn_err(m,Ze2n)[sdj]))
    print("=== Robustness to feature noise (source-disjoint ROC) ===")
    print(f"  {'noise σ':>8} {'Mahalanobis':>14} {'Attention':>12}")
    for L in levels:
        print(f"  {L:>8.2f} {np.mean(res[L]['maha']):>14.3f} {np.mean(res[L]['attn']):>12.3f}")
    return res
robustness_noise()

=== Robustness to feature noise (source-disjoint ROC) ===
   noise σ    Mahalanobis    Attention
      0.00          0.907        0.976
      0.05          0.904        0.974
      0.10          0.898        0.970
      0.20          0.878        0.945
      0.30          0.855        0.909


{0.0: {'maha': [np.float64(0.9052932189749289),
   np.float64(0.9054166614200893),
   np.float64(0.9048476714338611),
   np.float64(0.9105501789370423),
   np.float64(0.9088436755216953)],
  'attn': [np.float64(0.9744082123715654),
   np.float64(0.9766759739354713),
   np.float64(0.9741310750845581),
   np.float64(0.9732394108012081),
   np.float64(0.9795562653095391)]},
 0.05: {'maha': [np.float64(0.9033557562566765),
   np.float64(0.9018490124342338),
   np.float64(0.9012719916711776),
   np.float64(0.9077117167714277),
   np.float64(0.90570068688954)],
  'attn': [np.float64(0.9722069509091515),
   np.float64(0.9730348350746645),
   np.float64(0.9709012749748684),
   np.float64(0.9743708854901681),
   np.float64(0.9773366467529185)]},
 0.1: {'maha': [np.float64(0.897958867597854),
   np.float64(0.8955574302380529),
   np.float64(0.8950084510580852),
   np.float64(0.9018068413131579),
   np.float64(0.8995882676068196)],
  'attn': [np.float64(0.9688113138681322),
   np.float64(0.970831

In [ ]:
# ---------- R1.2b: imbalance robustness (target subsampling) ----------
def robustness_imbalance(seeds=SEEDS, fracs=(1.0,0.5,0.25,0.10,0.05)):
    cfg=UAV_CFG; lab=cfg["label_col"]; normal=cfg["normal"]; tgt=TARGET; src=ID_CFG["src"]
    df=load(cfg); df=pd.concat([df,source_rate_features(df,ID_CFG)],axis=1)
    res={f:[] for f in fracs}
    for seed in seeds:
        tr,valk,test_=loaco_split(df,lab,tgt,seed); test_=test_.reset_index(drop=True)
        ytr=tr[lab].astype(str).values; ye=test_[lab].astype(str).values
        is_t=(ye==tgt); bmask=(ytr==normal)
        seen=is_t & test_[src].astype(str).isin(set(tr[src].astype(str))).values; sdj=~seen
        A=tr[RATE].reset_index(drop=True); B=test_[RATE].reset_index(drop=True)
        Za,Ze=_Z(A,B); mu,P=fit_mahal(Za[bmask]); s=mahal(Ze,mu,P)
        rng=np.random.default_rng(seed)
        idx=np.where(sdj)[0]; tgt_idx=idx[is_t[sdj]]; neg_idx=idx[~is_t[sdj]]
        for f in fracs:
            keep_t=rng.choice(tgt_idx, max(1,int(len(tgt_idx)*f)), replace=False)
            keep=np.concatenate([keep_t, neg_idx])
            res[f].append(roc_auc_score(is_t[keep].astype(int), s[keep]))
    print("=== Robustness to target imbalance (Mahalanobis, source-disjoint ROC) ===")
    print(f"  {'Sybil frac':>10} {'ROC':>8}")
    for f in fracs: print(f"  {f:>10.3f} {np.mean(res[f]):>8.3f}")
    print(">>> ROC threshold-bağımsız (AUC), prevalence-invariant beklenir.")
    return res
robustness_imbalance()

=== Robustness to target imbalance (Mahalanobis, source-disjoint ROC) ===
  Sybil frac      ROC
       1.000    0.907
       0.500    0.907
       0.250    0.907
       0.100    0.905
       0.050    0.909
>>> ROC threshold-bağımsız (AUC), prevalence-invariant beklenir.


{1.0: [np.float64(0.9052932189749289),
  np.float64(0.9054166614200893),
  np.float64(0.9048476714338611),
  np.float64(0.9105501789370423),
  np.float64(0.9088436755216953)],
 0.5: [np.float64(0.9049940521615619),
  np.float64(0.905324503011036),
  np.float64(0.9061838828223548),
  np.float64(0.909136908352738),
  np.float64(0.9095660145976462)],
 0.25: [np.float64(0.905352370364049),
  np.float64(0.9043049675454848),
  np.float64(0.903886538990251),
  np.float64(0.9116350709056614),
  np.float64(0.9095468868417305)],
 0.1: [np.float64(0.9036555307995547),
  np.float64(0.9031419331649695),
  np.float64(0.9034977130796396),
  np.float64(0.9109680977549001),
  np.float64(0.9047936545917009)],
 0.05: [np.float64(0.9038988119766482),
  np.float64(0.9121557408204604),
  np.float64(0.9071608159384941),
  np.float64(0.9094994676869615),
  np.float64(0.911620862155196)]}